# 🔬 ResearchAI — Colab Demo (ngrok)

Run an **agentic, self-correcting RAG** research assistant entirely inside Colab and
get a **public URL** you can open on any device — perfect for a live competition demo.

**Stack:** Streamlit → FastAPI → LangChain → Groq API (Llama 3.1 / 3.3) ·
ChromaDB · BAAI BGE embeddings · hybrid BM25+dense + RRF · BGE reranker · PyMuPDF · Self-RAG · knowledge graph.

### ⏱️ Run order (top to bottom, ~3–5 min first time)
`GPU check (optional) → install → get code → configure → start API → warm up → launch + ngrok`

> The LLM runs remotely on the **Groq API** (free) — no GPU or multi-GB model
> download required. A GPU only speeds up the local embeddings/reranker if you have one.
> Grab a free Groq key at https://console.groq.com/keys and a free ngrok token at
> https://dashboard.ngrok.com/get-started/your-authtoken — paste both in the **Configuration** cell.

## 0 · GPU check (optional)
Only the local embeddings and BGE reranker use a GPU here — the LLM itself runs on the
Groq API. Skip this if Colab didn't give you one; everything still works on CPU, just a
bit slower to embed/rerank.

In [ ]:
import subprocess
r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
    capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout.strip())
else:
    print("⚠️ No GPU detected. Runtime → Change runtime type → GPU (T4), then rerun.")

## 1 · Get the ResearchAI code

Two ways — pick one by toggling `USE_GIT`:

* **Upload the zip** (default): run the cell, then choose `research-ai.zip` from your computer.
* **Clone a repo** (best for repeat demos): set `USE_GIT = True` and your `GIT_URL`.

In [ ]:
import os, io, zipfile, glob

USE_GIT = False                                   # ← set True to clone instead of upload
GIT_URL = "https://github.com/<your-username>/research-ai.git"

if USE_GIT:
    !git clone $GIT_URL research-ai
else:
    from google.colab import files
    print("Choose research-ai.zip …")
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(io.BytesIO(up[name])) as z:
        z.extractall(".")

# Enter the project folder (handles either layout)
root = "research-ai" if os.path.isdir("research-ai") else \
       next((os.path.dirname(p) for p in glob.glob("**/backend/main.py", recursive=True)), ".")
os.chdir(root)
print("📂 Working dir:", os.getcwd())
assert os.path.isfile("backend/main.py"), "Couldn't find the project — re-check the upload/clone."
print("✅ Code ready:", sorted(os.listdir()))

## 2 · Install Python dependencies
Colab already ships a CUDA build of PyTorch, so we install everything **except** torch
(and skip the optional `FlagEmbedding`) for a fast, conflict-free setup. The LLM itself
needs no local install at all — it's called over HTTP (Groq by default; `huggingface_hub`
is also installed in case you switch `LLM_BACKEND` to `hf_api`).

In [ ]:
pkgs = (
    "langchain langchain-core langchain-community "
    "langchain-chroma langchain-huggingface langchain-text-splitters "
    "huggingface_hub chromadb sentence-transformers rank-bm25 PyMuPDF "
    "networkx pyvis fastapi uvicorn[standard] python-multipart "
    "pydantic pydantic-settings streamlit requests pyngrok"
)
!pip install -q {pkgs}
print("✅ Dependencies installed")

## 3 · Configuration  ⚙️
Colab **form** — pick your model, add your Groq API key, and paste your ngrok token,
then run to write `.env`.

* **`GROQ_API_KEY`** → free at https://console.groq.com/keys.
  Prefer storing it as a Colab **secret** named `GROQ_API_KEY` (🔑 icon in the left sidebar)
  instead of pasting it into the form.
* **`llama-3.1-8b-instant`** → fastest, snappiest live demo.
* **`llama-3.3-70b-versatile`** → project default, higher quality, still fast.

In [ ]:
#@title Configuration { display-mode: "form" }
GROQ_MODEL = "llama-3.3-70b-versatile" #@param ["llama-3.3-70b-versatile", "llama-3.1-8b-instant"]
GROQ_API_KEY = "" #@param {type:"string"}
EMBED_DEVICE = "cuda" #@param ["cuda", "cpu"]
NGROK_AUTHTOKEN = "" #@param {type:"string"}
RERANK_TOP_K = 5 #@param {type:"integer"}
SELF_RAG_MAX_RETRIES = 2 #@param {type:"integer"}

if not GROQ_API_KEY:
    try:
        from google.colab import userdata
        GROQ_API_KEY = userdata.get("GROQ_API_KEY") or ""
    except Exception:
        pass

env = f"""LLM_BACKEND=groq
GROQ_API_KEY={GROQ_API_KEY}
GROQ_MODEL={GROQ_MODEL}
LLM_TEMPERATURE=0.1
LLM_MAX_TOKENS=1024
LLM_TIMEOUT=120
EMBED_MODEL=BAAI/bge-base-en-v1.5
RERANKER_MODEL=BAAI/bge-reranker-base
EMBED_DEVICE={EMBED_DEVICE}
DENSE_TOP_K=20
BM25_TOP_K=20
RERANK_TOP_K={RERANK_TOP_K}
MULTI_QUERY_N=3
SELF_RAG_MAX_RETRIES={SELF_RAG_MAX_RETRIES}
API_HOST=0.0.0.0
API_PORT=8000
BACKEND_URL=http://localhost:8000
NGROK_AUTHTOKEN={NGROK_AUTHTOKEN}
"""
open(".env", "w").write(env)
print(".env written (key masked):\n")
print(env.replace(GROQ_API_KEY, "***") if GROQ_API_KEY else env)
if not GROQ_API_KEY:
    print("⚠️  No GROQ_API_KEY set — get a free one at https://console.groq.com/keys")
if not NGROK_AUTHTOKEN:
    print("ℹ️ No ngrok token set — the tunnel may work but can be less stable/time-limited.")

## 4 · Start the FastAPI backend
Runs the API on `:8000` in the background and waits for `/health`.

In [ ]:
subprocess.run(["pkill", "-f", "uvicorn"], check=False)
time.sleep(2)
backend_proc = subprocess.Popen(
    ["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("logs/backend.log", "w"), stderr=subprocess.STDOUT)

health = None
for _ in range(90):
    try:
        health = requests.get("http://localhost:8000/health", timeout=3).json(); break
    except Exception:
        time.sleep(2)
print("✅ Backend health:", health if health else "NOT UP — check logs/backend.log")

## 5 · Warm up + end-to-end smoke test  🔥
This is the **demo-tuning** step: it ingests a tiny sample PDF and asks one question,
which forces the backend to load the **embeddings, BM25 index, and BGE reranker** and warms
up the **Groq API** connection *now* — so your first question on stage returns
quickly instead of cold-starting. It also proves the whole grounded-Q&A pipeline works
before you go live.

In [ ]:
import fitz, time, requests
from pathlib import Path

# tiny 1-page paper-like PDF
p = Path("sample.pdf")
doc = fitz.open(); pg = doc.new_page()
pg.insert_text((72, 72),
    "Introduction\n\nThis paper proposes a hybrid retrieval method that combines "
    "BM25 with dense BGE embeddings, fused via reciprocal rank fusion and reranked "
    "with a cross-encoder. We evaluate on the SQuAD dataset and optimize a "
    "cross-entropy loss. Exact-match accuracy improves by 4.2 points over a dense-only "
    "baseline. E = mc^2 is unrelated but tests equation detection.")
doc.save(p); doc.close()

with open(p, "rb") as f:
    ing = requests.post("http://localhost:8000/ingest",
                        files={"file": ("sample.pdf", f, "application/pdf")},
                        timeout=600).json()
print("📄 Ingested:", ing)

t = time.time()
ans = requests.post("http://localhost:8000/ask",
                    json={"question": "What datasets and loss function are used, and how much did accuracy improve?"},
                    timeout=600).json()
print(f"\n⏱️  Answered in {time.time()-t:.1f}s (this warm-up pays for itself on stage)\n")
print(ans["answer"])
v = ans.get("verification", {})
print(f"\n🔒 Grounding score: {v.get('grounding_score')}  "
      f"({v.get('supported')}/{v.get('total')} claims supported)")
print("🔁 Self-corrected:", ans.get("self_corrected"))

## 6 · Launch the UI + open the public ngrok URL  🌍
Starts Streamlit on `:8501` and opens a public tunnel. **Click the printed URL** —
that's your live demo. (Only the frontend is tunnelled; it talks to the API over
localhost, so a single free ngrok tunnel is all you need.)

In [ ]:
from pyngrok import ngrok, conf

if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN

# clean any previous tunnels / streamlit
ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"], check=False)
time.sleep(2)

streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "frontend/app.py",
     "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    stdout=open("logs/streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)

public_url = ngrok.connect(8501, "http").public_url
print("=" * 64)
print("🌍  OPEN YOUR DEMO:", public_url)
print("=" * 64)
print("Backend API docs (local): http://localhost:8000/docs")
print("If the page is blank, wait ~10s and refresh (Streamlit is still booting).")

## 🎬 Demo script (what to click)

1. **Upload** 2–3 papers in the sidebar — call out the page/chunk counts.
2. **Q&A tab** — ask a hard, specific question. Highlight the **grounding score**,
   the ✅/⚠️ per-claim badges, the **page citations**, and the **Self-RAG trace**.
3. **Agent tab** — type *"compare the two papers' methods"* → it **auto-routes** to the
   comparison tool. This is the "agentic" wow moment.
4. **Gaps tab** — surface concrete research gaps with evidence + suggested directions.
5. **Knowledge Graph tab** — extract and render the interactive graph.
6. Open the ngrok URL **on your phone** to prove it's genuinely live.

**One-liner:** *"An agentic RAG that doesn't just answer — it retrieves with hybrid
search, reranks, self-corrects, and proves every claim against the source page."*

---
### 🩺 Quick troubleshooting
| Symptom | Fix |
|---|---|
| UI says "Backend not reachable" | Re-run **cell 4**, check `logs/backend.log` |
| Blank ngrok page | Wait ~10s, refresh; check `logs/streamlit.log` |
| Very slow first answer | You skipped **cell 5** — run the warm-up |
| LLM/auth errors | Check `GROQ_API_KEY` is set and valid; if a model id 404s, Groq may have retired it — see https://console.groq.com/docs/models |
| Out-of-memory (embeddings/reranker) | Set `EMBED_DEVICE=cpu`, or lower `DENSE_TOP_K`/`BM25_TOP_K` |

## 🧹 (Optional) Tear down
Stops all background processes and closes the tunnel — run when you're done.

In [ ]:
from pyngrok import ngrok
for name in ["streamlit_proc", "backend_proc"]:
    try:
        globals()[name].terminate()
    except Exception:
        pass
ngrok.kill()
for pat in ["streamlit", "uvicorn"]:
    subprocess.run(["pkill", "-f", pat], check=False)
print("🛑 All services stopped.")